# Enterprise Orchestration Lab — RL Training

Manual REINFORCE loop with environment-grounded rewards. Colab T4 GPU required.

> If numpy errors occur: **Runtime > Factory reset runtime** first.

---
## 1. Install

In [ ]:
numpy_ok = True
try:
    import numpy as np; from numpy._core.umath import isalpha
    print(f'numpy {np.__version__} OK')
except Exception:
    numpy_ok = False; import subprocess
    subprocess.check_call(['pip','install','-q','numpy==1.26.4'])
    print('numpy fixed. Click Runtime > Restart session, then re-run.')
if numpy_ok:
    import subprocess
    subprocess.check_call(['pip','install','-q','trl','peft','accelerate','bitsandbytes','datasets','matplotlib'])
    subprocess.check_call(['pip','install','-q','openenv-core'])
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from peft import get_peft_model, LoraConfig
    print(f'torch {torch.__version__}, CUDA: {torch.cuda.is_available()}')
    print('All imports OK.')

---
## 2. Setup Environment

In [ ]:
import os, sys, json, re, random
if not os.path.exists('scaler-final'):
    !git clone https://github.com/redhatsam09/scaler-final.git
if not os.getcwd().endswith('scaler-final'): os.chdir('scaler-final')
!pip install -q -r requirements.txt
if '.' not in sys.path: sys.path.insert(0, '.')
from src.environment import DataCleaningEnv
from src.graders import EnterpriseOrchestrationGrader, MissingValuesGrader
from src.models import Action
print('Environment loaded OK.')

---
## 3. Reward Function

In [ ]:
VALID_ACTIONS = {'analyze','impute','deduplicate','validate','report_findings','delegate','resolve_alert','reconcile_apps','oversight_review','inspect_actor','audit_records','request_policy_clarification'}
SYSTEM_PROMPT = 'You are an enterprise workflow orchestrator. Output a JSON action with: action_type, target_columns (list), parameters (dict), reasoning (string, 15+ chars). Respond ONLY with JSON.'

def build_prompt(obs):
    return f"{SYSTEM_PROMPT}\n\nState: {obs.natural_language_observation[:500]}\nStep: {obs.step_count}\nKPIs: {json.dumps(obs.kpi_snapshot)}\nAction?"

def compute_reward(completion):
    match = re.search(r'\{.*\}', completion, re.DOTALL)
    if not match: return -1.0
    try: data = json.loads(match.group())
    except: return -0.5
    if not isinstance(data, dict): return -0.3
    has_keys = sum(1 for k in ['action_type','target_columns','parameters','reasoning'] if k in data)
    if has_keys < 4: return -0.3 + has_keys * 0.1
    if data.get('action_type','') not in VALID_ACTIONS: return 0.0
    reasoning_len = len(data.get('reasoning',''))
    r_bonus = min(0.2, reasoning_len / 100.0 * 0.2)
    try:
        env = DataCleaningEnv(seed=42)
        obs = env.reset(task_id='task_enterprise_orchestration', seed=random.randint(0,9999))
        action = Action(action_type=data['action_type'], target_columns=data.get('target_columns',[]),
                        parameters=data.get('parameters',{}), reasoning=data.get('reasoning','auto')[:200])
        _, rw, _, _ = env.step(action)
        return 0.3 + 0.5 * rw.value + r_bonus
    except: return 0.2 + r_bonus

# Test
for label, comp in [('no json','hello'), ('valid',json.dumps({'action_type':'analyze','target_columns':['id'],'parameters':{},'reasoning':'Check data quality first'}))]:
    print(f'  {label}: {compute_reward(comp):+.2f}')

---
## 4. Load Model

In [ ]:
MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb_config, device_map='auto')
model = get_peft_model(model, LoraConfig(r=16, lora_alpha=16, lora_dropout=0, task_type='CAUSAL_LM',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj']))
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
model.print_trainable_parameters()

---
## 5. Generate Prompts

In [ ]:
prompt_env = DataCleaningEnv(seed=42)
task_ids = ['task_enterprise_orchestration','task_missing_values','task_duplicate_handling','task_complex_validation']
prompts = []
for i in range(40):
    obs = prompt_env.reset(task_id=task_ids[i%4], seed=2000+i)
    prompts.append(build_prompt(obs))
print(f'{len(prompts)} prompts ready')

---
## 6. REINFORCE Training Loop

Manual loop: generate completion, compute reward, get log-probs, compute policy gradient loss. This guarantees non-zero gradients.

In [ ]:
from torch.cuda.amp import autocast
import time

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
NUM_STEPS = 30
loss_history = []
reward_history = []
model.train()

print('Step | Loss       | Reward  | Time')
print('-----|------------|---------|------')

for step in range(NUM_STEPS):
    t0 = time.time()
    prompt = prompts[step % len(prompts)]
    
    # Encode prompt
    enc = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=1024).to(model.device)
    prompt_len = enc['input_ids'].shape[1]
    
    # Generate completion
    with torch.no_grad():
        gen_out = model.generate(**enc, max_new_tokens=200, temperature=1.0, do_sample=True, top_p=0.95)
    
    gen_ids = gen_out[0]  # full sequence (prompt + completion)
    comp_ids = gen_ids[prompt_len:]  # completion only
    completion = tokenizer.decode(comp_ids, skip_special_tokens=True)
    
    # Compute reward
    reward = compute_reward(completion)
    reward_history.append(reward)
    
    # Compute log-probs of the generated tokens under current policy
    with autocast(dtype=torch.float16):
        outputs = model(gen_ids.unsqueeze(0))
        logits = outputs.logits[0, prompt_len-1:-1, :]  # logits for completion positions
        log_probs = torch.nn.functional.log_softmax(logits.float(), dim=-1)
        token_log_probs = log_probs.gather(1, comp_ids.unsqueeze(1)).squeeze(1)
        mean_log_prob = token_log_probs.mean()
    
    # REINFORCE loss: -reward * log_prob
    # Normalize reward to have baseline of 0
    baseline = sum(reward_history) / len(reward_history) if reward_history else 0
    advantage = reward - baseline
    loss = -advantage * mean_log_prob
    
    # Backward + update
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    
    loss_val = loss.item()
    loss_history.append(loss_val)
    elapsed = time.time() - t0
    
    print(f'{step+1:4d} | {loss_val:+10.6f} | {reward:+.4f} | {elapsed:.1f}s')

print(f'\nTraining complete. Mean reward: {sum(reward_history)/len(reward_history):+.4f}')

---
## 7. Training Curves

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib
matplotlib.rcParams.update({'font.size':11,'figure.facecolor':'#0f172a','axes.facecolor':'#1e293b',
    'axes.edgecolor':'#334155','text.color':'#f8fafc','axes.labelcolor':'#94a3b8',
    'xtick.color':'#64748b','ytick.color':'#64748b'})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(range(1,len(loss_history)+1), loss_history, color='#38bdf8', lw=2, marker='o', ms=3)
axes[0].fill_between(range(1,len(loss_history)+1), loss_history, alpha=0.15, color='#38bdf8')
axes[0].set_xlabel('Step'); axes[0].set_ylabel('Loss')
axes[0].set_title('REINFORCE Training Loss', color='#38bdf8', fontweight='bold')
axes[0].grid(True, alpha=0.2, color='#475569')

# Reward with trend
axes[1].plot(range(1,len(reward_history)+1), reward_history, color='#22c55e', lw=2, marker='o', ms=3)
axes[1].fill_between(range(1,len(reward_history)+1), reward_history, alpha=0.15, color='#22c55e')
if len(reward_history) > 3:
    z = np.polyfit(range(len(reward_history)), reward_history, 1)
    axes[1].plot(range(1,len(reward_history)+1), np.poly1d(z)(range(len(reward_history))),
                 '--', color='#f59e0b', lw=1.5, label=f'Trend (slope={z[0]:.4f})')
    axes[1].legend(facecolor='#1e293b', edgecolor='#334155')
axes[1].axhline(y=0, color='#64748b', ls='--', alpha=0.4)
axes[1].set_xlabel('Step'); axes[1].set_ylabel('Reward')
axes[1].set_title('Per-Step Reward', color='#22c55e', fontweight='bold')
axes[1].grid(True, alpha=0.2, color='#475569')

plt.tight_layout()
plt.savefig('artifacts/training_curves.svg', facecolor='#0f172a')
plt.savefig('artifacts/training_curves.png', facecolor='#0f172a', dpi=150)
plt.show()

---
## 8. Evaluate

In [ ]:
eval_rewards = []
model.eval()
for i in range(12):
    obs = DataCleaningEnv(seed=99).reset(task_id=task_ids[i%4], seed=5000+i)
    prompt = build_prompt(obs)
    enc = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=1024).to(model.device)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=200, temperature=0.7, do_sample=True)
    comp = tokenizer.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True)
    r = compute_reward(comp)
    eval_rewards.append(r)
    print(f'Episode {i+1:2d} | {task_ids[i%4]:36s} | reward={r:+.2f}')

mean_r = sum(eval_rewards)/len(eval_rewards)
fig, ax = plt.subplots(figsize=(8,4))
ax.bar(range(1,13), eval_rewards, color=['#22c55e' if r>0 else '#ef4444' for r in eval_rewards])
ax.axhline(y=0, color='#64748b', ls='--', alpha=0.4)
ax.set_xlabel('Episode'); ax.set_ylabel('Reward')
ax.set_title(f'Eval Rewards (mean={mean_r:+.2f})', color='#38bdf8', fontweight='bold')
ax.grid(True, alpha=0.2, color='#475569')
plt.tight_layout()
plt.savefig('artifacts/eval_results.svg', facecolor='#0f172a')
plt.show()

---
## 9. Save Metrics

In [ ]:
from pathlib import Path
metrics = {'model': MODEL_NAME, 'method': 'REINFORCE', 'steps': NUM_STEPS,
    'loss_history': [round(l,6) for l in loss_history],
    'reward_history': [round(r,4) for r in reward_history],
    'eval_mean': round(mean_r,4), 'eval_per_episode': [round(r,4) for r in eval_rewards]}
Path('artifacts').mkdir(exist_ok=True)
Path('artifacts/grpo_training_metrics.json').write_text(json.dumps(metrics, indent=2))
print(json.dumps(metrics, indent=2))